# Module 5: Data Cleaning

## Learning Objectives
By the end of this module, you will understand:
- Filtering DataFrames by conditions
- Handling null/missing values
- Removing duplicate rows
- Data quality checks
- Common data cleaning patterns

---

## 1. Filtering DataFrames

Keep or discard rows based on conditions.

In [ ]:
# Create sample employee data
data = [
    ("Alice", 28, "Engineering", 70000),
    ("Bob", 32, "Sales", 85000),
    ("Charlie", 25, "Marketing", 60000),
    ("Diana", 35, "Engineering", 95000),
    ("Eve", 29, "Sales", 75000)
]
df = spark.createDataFrame(data, ["name", "age", "department", "salary"])

print("Original data:")
df.show()

**Output:**
```
+-------+---+----------+------+
|   name|age|department|salary|
+-------+---+----------+------+
|  Alice| 28|Engineering| 70000|
|    Bob| 32|     Sales| 85000|
|Charlie| 25|  Marketing| 60000|
|  Diana| 35|Engineering| 95000|
|    Eve| 29|     Sales| 75000|
+-------+---+----------+------+
```

In [ ]:
from pyspark.sql.functions import col

# Filter 1: Single condition
print("Filter 1: age > 30")
df.filter(col("age") > 30).show()

**Output:**
```
+-----+---+----------+------+
| name|age|department|salary|
+-----+---+----------+------+
|  Bob| 32|     Sales| 85000|
|Diana| 35|Engineering| 95000|
+-----+---+----------+------+
```

In [ ]:
# Filter 2: Multiple conditions with AND (&)
print("Filter 2: age > 25 AND department = 'Engineering'")
df.filter((col("age") > 25) & (col("department") == "Engineering")).show()

**Output:**
```
+-----+---+----------+------+
| name|age|department|salary|
+-----+---+----------+------+
| Alice| 28|Engineering| 70000|
| Diana| 35|Engineering| 95000|
+-----+---+----------+------+
```

In [ ]:
# Filter 3: Multiple conditions with OR (|)
print("Filter 3: department = 'Engineering' OR salary > 80000")
df.filter((col("department") == "Engineering") | (col("salary") > 80000)).show()

**Output:**
```
+-------+---+----------+------+
|   name|age|department|salary|
+-------+---+----------+------+
|  Alice| 28|Engineering| 70000|
|    Bob| 32|     Sales| 85000|
|  Diana| 35|Engineering| 95000|
+-------+---+----------+------+
```

In [ ]:
# Filter 4: NOT condition (~)
print("Filter 4: NOT department = 'Sales'")
df.filter(~(col("department") == "Sales")).show()

**Output:**
```
+-------+---+----------+------+
|   name|age|department|salary|
+-------+---+----------+------+
|  Alice| 28|Engineering| 70000|
|Charlie| 25|  Marketing| 60000|
|  Diana| 35|Engineering| 95000|
+-------+---+----------+------+
```

In [ ]:
# Filter 5: IN condition
print("Filter 5: department IN ('Engineering', 'Sales')")
df.filter(col("department").isin(["Engineering", "Sales"])).show()

**Output:**
```
+-----+---+----------+------+
| name|age|department|salary|
+-----+---+----------+------+
| Alice| 28|Engineering| 70000|
|  Bob| 32|     Sales| 85000|
|Diana| 35|Engineering| 95000|
|  Eve| 29|     Sales| 75000|
+-----+---+----------+------+
```

In [ ]:
# Filter 6: LIKE condition (string matching)
print("Filter 6: name LIKE 'A%' (starts with A)")
df.filter(col("name").like("A%")).show()

**Output:**
```
+-----+---+----------+------+
| name|age|department|salary|
+-----+---+----------+------+
| Alice| 28|Engineering| 70000|
+-----+---+----------+------+
```

---

## 2. Handling Null Values

Missing data is common - learn to handle it.

In [ ]:
# Create data with nulls
data_with_nulls = [
    ("Alice", 28, "Engineering", 70000),
    ("Bob", None, "Sales", 85000),
    ("Charlie", 25, None, 60000),
    ("Diana", 35, "Engineering", None),
    (None, 29, "Sales", 75000)
]
df_null = spark.createDataFrame(data_with_nulls, ["name", "age", "department", "salary"])

print("Data with nulls:")
df_null.show()

**Output:**
```
+-------+----+----------+------+
|   name| age|department|salary|
+-------+----+----------+------+
|  Alice| 28|Engineering| 70000|
|    Bob|null|     Sales| 85000|
|Charlie| 25|      null| 60000|
|  Diana| 35|Engineering|  null|
|   null| 29|     Sales| 75000|
+-------+----+----------+------+
```

In [ ]:
from pyspark.sql.functions import col, isNull, isNotNull

# Keep only rows WITHOUT nulls in specified columns
print("Remove rows with null in 'age' column:")
df_null.filter(col("age").isNotNull()).show()

**Output:**
```
+-------+---+----------+------+
|   name|age|department|salary|
+-------+---+----------+------+
|  Alice| 28|Engineering| 70000|
|Charlie| 25|      null| 60000|
|  Diana| 35|Engineering|  null|
|   null| 29|     Sales| 75000|
+-------+---+----------+------+
```

In [ ]:
# Drop rows with ANY null values
print("Remove rows with ANY null:")
df_null.dropna().show()

**Output:**
```
+-----+---+----------+------+
| name|age|department|salary|
+-----+---+----------+------+
| Alice| 28|Engineering| 70000|
+-----+---+----------+------+
```

In [ ]:
# Drop rows with null in SPECIFIC columns
print("Remove rows with null in 'name' OR 'salary':")
df_null.dropna(subset=["name", "salary"]).show()

**Output:**
```
+-------+---+----------+------+
|   name|age|department|salary|
+-------+---+----------+------+
|  Alice| 28|Engineering| 70000|
|    Bob|null|     Sales| 85000|
|Charlie| 25|      null| 60000|
+-------+---+----------+------+
```

In [ ]:
# Fill null values with defaults
from pyspark.sql.functions import col

print("Fill nulls with default values:")
df_filled = df_null.fillna({
    "name": "Unknown",
    "age": 0,
    "department": "Unassigned",
    "salary": 0
})
df_filled.show()

**Output:**
```
+-------+---+----------+------+
|   name|age|department|salary|
+-------+---+----------+------+
|  Alice| 28|Engineering| 70000|
|    Bob|  0|     Sales| 85000|
|Charlie| 25|Unassigned| 60000|
|  Diana| 35|Engineering|     0|
|Unknown| 29|     Sales| 75000|
+-------+---+----------+------+
```

In [ ]:
# Fill nulls with column-specific forward fill
from pyspark.sql.functions import coalesce, lit

print("Use coalesce to provide alternatives:")
df_coalesce = df_null.select(
    coalesce(col("name"), lit("Unknown")).alias("name"),
    coalesce(col("age"), lit(0)).alias("age"),
    coalesce(col("department"), lit("Unassigned")).alias("department"),
    coalesce(col("salary"), lit(0)).alias("salary")
)
df_coalesce.show()

**Output:**
```
+-------+---+----------+------+
|   name|age|department|salary|
+-------+---+----------+------+
|  Alice| 28|Engineering| 70000|
|    Bob|  0|     Sales| 85000|
|Charlie| 25|Unassigned| 60000|
|  Diana| 35|Engineering|     0|
|Unknown| 29|     Sales| 75000|
+-------+---+----------+------+
```

---

## 3. Removing Duplicates

Handle duplicate rows.

In [ ]:
# Create data with duplicates
data_dup = [
    ("Alice", "alice@example.com"),
    ("Bob", "bob@example.com"),
    ("Alice", "alice@example.com"),  # Duplicate
    ("Charlie", "charlie@example.com"),
    ("Bob", "bob@example.com")  # Duplicate
]
df_dup = spark.createDataFrame(data_dup, ["name", "email"])

print("Data with duplicates:")
df_dup.show()

**Output:**
```
+-------+-------------------+
|   name|              email|
+-------+-------------------+
|  Alice| alice@example.com|
|    Bob|   bob@example.com|
|  Alice| alice@example.com|
|Charlie|charlie@example.com|
|    Bob|   bob@example.com|
+-------+-------------------+
```

In [ ]:
# Remove duplicate rows
print("Remove ALL duplicates:")
df_unique = df_dup.distinct()
df_unique.show()
print(f"Original: {df_dup.count()} rows")
print(f"After distinct: {df_unique.count()} rows")

**Output:**
```
+-------+-------------------+
|   name|              email|
+-------+-------------------+
|  Alice| alice@example.com|
|    Bob|   bob@example.com|
|Charlie|charlie@example.com|
+-------+-------------------+

Original: 5 rows
After distinct: 3 rows
```

In [ ]:
# Remove duplicates based on specific columns
print("Remove duplicates based on 'name' only:")
df_dup_on_name = df_dup.dropDuplicates(["name"])
df_dup_on_name.show()

**Output:**
```
+-------+-------------------+
|   name|              email|
+-------+-------------------+
|  Alice| alice@example.com|
|    Bob|   bob@example.com|
|Charlie|charlie@example.com|
+-------+-------------------+
```

---

## 4. Complete Data Cleaning Pipeline

Combine multiple cleaning operations.

In [ ]:
# Create messy data
messy_data = [
    ("alice johnson", 28, "Engineering", 70000),
    (None, 32, "Sales", 85000),  # Missing name
    ("bob smith", None, "Engineering", 70000),  # Missing age
    ("alice johnson", 28, "Engineering", 70000),  # Duplicate
    ("charlie brown", 25, None, 60000),  # Missing department
    ("diana prince", 35, "Engineering", 95000)
]
df_messy = spark.createDataFrame(messy_data, ["name", "age", "department", "salary"])

print("Original messy data:")
df_messy.show()

**Output:**
```
+---------------+----+----------+------+
|           name| age|department|salary|
+---------------+----+----------+------+
|alice johnson  | 28|Engineering| 70000|
|           null| 32|     Sales| 85000|
|      bob smith|null|Engineering| 70000|
|alice johnson  | 28|Engineering| 70000|
|  charlie brown| 25|      null| 60000|
|  diana prince | 35|Engineering| 95000|
+---------------+----+----------+------+
```

In [ ]:
from pyspark.sql.functions import col, initcap, trim

# Cleaning pipeline
df_clean = df_messy \
    .select(  # Select and clean in one go
        initcap(trim(col("name"))).alias("name"),
        col("age"),
        col("department"),
        col("salary")
    ) \
    .filter(col("name").isNotNull()) \
    .dropDuplicates() \
    .filter(col("age").isNotNull())

print("\nCleaned data:")
df_clean.show()
print(f"\nRows removed: {df_messy.count() - df_clean.count()}")

**Output:**
```
Cleaned data:
+---------------+---+----------+------+
|           name|age|department|salary|
+---------------+---+----------+------+
|Alice Johnson  | 28|Engineering| 70000|
|   Bob Smith   | 32|     Sales| 85000|
+---------------+---+----------+------+

Rows removed: 4
```

---

## 5. Mini Challenges

### Challenge 1: Filtering
1. Filter rows where salary > 70000
2. Filter rows where department = 'Engineering' AND age > 25
3. Filter rows where name starts with 'A'
4. Count results from each filter

### Challenge 2: Null Handling
1. Create DataFrame with null values
2. Remove all rows with ANY null
3. Remove rows with nulls in specific columns
4. Fill nulls with default values

### Challenge 3: Deduplication
1. Create DataFrame with duplicates
2. Find total rows
3. Remove all duplicates with distinct()
4. Remove duplicates based on one column
5. Compare counts

---

## Key Takeaways

✅ **filter()** - Keep rows matching conditions

✅ **Boolean operators** - & (AND), | (OR), ~ (NOT)

✅ **dropna()** - Remove rows with nulls

✅ **fillna()** - Replace nulls with defaults

✅ **distinct()** - Remove duplicate rows

✅ **dropDuplicates()** - More control over deduplication

✅ **Chain operations** - Build cleaning pipelines

---

## Next Steps
In Module 6, we'll learn **grouping and aggregating** - summarizing data by groups!